# Custo Frete — Relatorio de Receitas ATUA

Gera o arquivo de **Custo Frete** no layout `FECHAMENTO_ODBC` a partir do mesmo arquivo de CTRCs usado para as Receitas (`Relatorio_Receitas_Sistema-ATUA_{MM}.xls`).

Ajuste o parametro `MES_REFERENCIA` na primeira celula de codigo (formato `MM/AAAA`).

A logica segue a especificacao do fechamento SAGI para *Saidas (Aplicacoes)*:

| Campo ATUA             | Campo FECHAMENTO_ODBC        | Observacao                              |
|------------------------|------------------------------|-----------------------------------------|
| `nr_ctrc`              | `titulo`                     | prefixo `CTRC-`                         |
| `dt_emissao`           | `data_nf` e `data_pagamento` |                                         |
| `nm_pessoa_filial`     | `filial` + hierarquia CC     | arvore de **despesa** 1.4 TRANSMOVE     |
| `nm_pessoa_motorista`  | `credor_forn_cli_func`       |                                         |
| `vl_frete_motorista`   | `valor_conta/nf/pago`        | valor **NEGATIVO**                      |
| —                      | `Origem`                     | `Saidas (Aplicacoes)`                   |
| —                      | `cod_conta`                  | `6.6.1 FRETE DE TERCEIROS` (provisorio) |

Colunas extras (fora do modelo ODBC): `C/D`, `F/V`, `D/I`, `Natureza`, `Tipo_C_D_DFC`.

Todas as linhas do relatorio de receitas sao mantidas (o total do fechamento deve ser igual ao total do arquivo ATUA). CTRCs com `vl_frete_motorista == 0` (motorista proprietario) aparecem no fechamento com valor zero.

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# ── Parametros ──────────────────────────────────────────────────────────────
MES_REFERENCIA = "05/2026"
ATUA_SUBPASTA = "Maio"
ARQUIVO_ENTRADA_NOME = "Relatorio_Receitas_Sistema-ATUA_05.xls"
MODELO_MES = "04/2026"
# ────────────────────────────────────────────────────────────────────────────

mes_num, ano = MES_REFERENCIA.split("/")

REFS_DIR = Path("../../02-Referencias")
ATUA_DIR = REFS_DIR / "ATUA"
ATUA_MES_DIR = (ATUA_DIR / ATUA_SUBPASTA) if ATUA_SUBPASTA else ATUA_DIR

if ARQUIVO_ENTRADA_NOME:
    ARQUIVO_ENTRADA = ATUA_MES_DIR / ARQUIVO_ENTRADA_NOME
else:
    ARQUIVO_ENTRADA = ATUA_MES_DIR / f"Relatorio_Receitas_Sistema-ATUA_{mes_num}.xls"

if MODELO_MES:
    _m_mes, _m_ano = MODELO_MES.split("/")
    ARQUIVO_MODELO = REFS_DIR / f"FECHAMENTO_ODBC_{_m_ano}_{_m_mes}.xlsx"
else:
    ARQUIVO_MODELO = REFS_DIR / f"FECHAMENTO_ODBC_{ano}_{mes_num}.xlsx"
if not ARQUIVO_MODELO.exists():
    _mods = sorted(REFS_DIR.glob("FECHAMENTO_ODBC_*.xlsx"))
    if not _mods:
        _mods = sorted((REFS_DIR / "Fechamento").glob("FECHAMENTO_ODBC_*.xlsx"))
    if not _mods:
        _mods = sorted((REFS_DIR / "outros").glob("FECHAMENTO_ODBC_*.xlsx"))
    if not _mods:
        _mods = [
            ATUA_MES_DIR / f"ATUA_despesas_fechamento_{mes_num}-{ano}.xlsx",
            ATUA_MES_DIR / f"ATUA_receitas_fechamento_{mes_num}-{ano}.xlsx",
        ]
        _mods = [p for p in _mods if p.exists()]
    if _mods:
        ARQUIVO_MODELO = _mods[-1]
        print(f"[AVISO] Modelo de {MES_REFERENCIA} nao encontrado; usando {ARQUIVO_MODELO.name}")
    else:
        raise FileNotFoundError(
            f"Nenhum FECHAMENTO_ODBC ou fechamento ATUA de referencia em {REFS_DIR.resolve()}"
        )

ARQUIVO_SAIDA = ATUA_MES_DIR / f"ATUA_custo_frete_fechamento_{mes_num}-{ano}.xlsx"

if not ARQUIVO_ENTRADA.exists():
    raise FileNotFoundError(f"Arquivo nao encontrado: {ARQUIVO_ENTRADA.resolve()}")

ATUA_MES_DIR.mkdir(parents=True, exist_ok=True)

raw_ctrc = pd.read_excel(
    ARQUIVO_ENTRADA,
    sheet_name=0,
    header=None,
    dtype=object,
    engine="xlrd",
    engine_kwargs={"ignore_workbook_corruption": True},
)

header_idx = None
for i in range(min(20, len(raw_ctrc))):
    vals = [str(v).strip().lower() for v in raw_ctrc.iloc[i].tolist() if pd.notna(v)]
    if "nm_pessoa_filial" in vals and "vl_frete_motorista" in vals:
        header_idx = i
        break
if header_idx is None:
    header_idx = 0

df_ctrc = raw_ctrc.iloc[header_idx + 1 :].copy()
df_ctrc.columns = [str(c).strip() for c in raw_ctrc.iloc[header_idx].tolist()]
df_ctrc = df_ctrc.reset_index(drop=True)
mask_vazia = df_ctrc.apply(
    lambda r: all(pd.isna(v) or str(v).strip() == "" for v in r.values), axis=1
)
df_ctrc = df_ctrc.loc[~mask_vazia].reset_index(drop=True)

import sys
sys.path.insert(0, str((Path.cwd().parent / "Utitlities").resolve()))
from fechamento_excel import normalizar_colunas_data

df_ctrc = normalizar_colunas_data(df_ctrc, ("dt_emissao",))

print(f"Header detectado na linha: {header_idx}")

def _to_float(v):
    if pd.isna(v):
        return 0.0
    try:
        return float(str(v).strip().replace(",", "."))
    except (ValueError, TypeError):
        return 0.0

df_ctrc["_valor"] = df_ctrc["vl_frete_motorista"].apply(_to_float)
df_custo = df_ctrc.copy().reset_index(drop=True)

n_zero = int((df_custo["_valor"] == 0).sum())
print(f"Entrada: {ARQUIVO_ENTRADA.resolve()}")
print(f"Total CTRCs: {len(df_ctrc)} | Com custo frete motorista: {len(df_ctrc) - n_zero} | Sem custo (zero): {n_zero}")
print()
df_custo[["nr_ctrc", "nm_pessoa_filial", "nm_pessoa_motorista", "vl_frete_empresa", "vl_frete_motorista"]].head(5)

[AVISO] Modelo de 05/2026 nao encontrado; usando FECHAMENTO_ODBC_2026_04.xlsx
Header detectado na linha: 1
Entrada: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\ATUA\Maio\Relatorio_Receitas_Sistema-ATUA_05.xls
Total CTRCs: 273 | Com custo frete motorista: 265 | Sem custo (zero): 8



,nr_ctrc,nm_pessoa_filial,nm_pessoa_motorista,vl_frete_empresa,vl_frete_motorista
0,1029,GSL PRUDENTE,ULISSES SANCHES LOPES,13472,10080
1,1030,GSL PRUDENTE,JOEL LEITE DE MORAIS,13472,10080
2,1031,GSL PRUDENTE,WILLIAN AMBROSIO,13472,10080
3,200,GSL DOURADOS,DAVID DE SOUZA FREITAS,12339.6,9175.6
4,201,GSL DOURADOS,JOSE ROBERTO VICENTE,20017.2,15251.2


## Centro de Custo — arvore de Despesa (1.4 TRANSMOVE)

O custo de frete pago ao motorista e uma **despesa** da GSL, portanto o CC segue a arvore `1 DESPESA > 1.4 TRANSMOVE GSL`, nao a arvore de receita (2.4).

| nm_pessoa_filial | n3     | n4       | Descricoes                                              |
|------------------|--------|----------|---------------------------------------------------------|
| GSL PRUDENTE     | 1.4.1  | 1.4.1.1  | TRANSMOVE GSL > PRESIDENTE PRUDENTE > TRANSPORTE        |
| GSL DOURADOS     | 1.4.2  | 1.4.2.1  | TRANSMOVE GSL > DOURADOS > TRANSPORTE                   |
| GSL MARINGA PR   | 1.4.3  | 1.4.3.1  | TRANSMOVE GSL > MARINGA > TRANSPORTE                    |

In [2]:
def _str(v) -> str:
    if pd.isna(v):
        return ""
    return str(v).strip()

MAPA_FILIAL_CUSTO = {
    "GSL PRUDENTE": {
        "n3_cod":  "1.4.1",
        "n3_desc": "PRESIDENTE PRUDENTE",
        "n4_cod":  "1.4.1.1",
        "n4_desc": "TRANSPORTE",
        "filial_saida": "GSL PRUDENTE",
    },
    "GSL DOURADOS": {
        "n3_cod":  "1.4.2",
        "n3_desc": "DOURADOS",
        "n4_cod":  "1.4.2.1",
        "n4_desc": "TRANSPORTE",
        "filial_saida": "GSL DOURADOS",
    },
    "GSL MARINGA PR": {
        "n3_cod":  "1.4.3",
        "n3_desc": "MARINGA",
        "n4_cod":  "1.4.3.1",
        "n4_desc": "TRANSPORTE",
        "filial_saida": "GSL MARINGA",
    },
}

def mapear_cc_custo(nm_filial) -> dict | None:
    """Retorna hierarquia SAGI completa para a filial, ou None se nao mapeada.

    Hierarquia de 3 niveis uteis (o prefixo numerico nao conta como nivel):
      n1: 1.4       -> TRANSMOVE GSL
      n2: 1.4.X     -> filial (ex: 1.4.1 PRESIDENTE PRUDENTE)
      n3: 1.4.X.Y   -> setor  (ex: 1.4.1.1 TRANSPORTE)
      n4: igual a n3 -> padrao do modelo FECHAMENTO_ODBC
    """
    info = MAPA_FILIAL_CUSTO.get(_str(nm_filial))
    if not info:
        return None
    return {
        "n1_cod":  "1.4",            "n1_desc": "TRANSMOVE GSL",
        "n2_cod":  info["n3_cod"],   "n2_desc": info["n3_desc"],
        "n3_cod":  info["n4_cod"],   "n3_desc": info["n4_desc"],
        "n4_cod":  info["n4_cod"],   "n4_desc": info["n4_desc"],
        "filial_saida": info["filial_saida"],
        "segmento": "TRANSMOVE GSL",
    }

filiais_sem_mapa = [_str(v) for v in df_custo["nm_pessoa_filial"].unique() if _str(v) not in MAPA_FILIAL_CUSTO]
if filiais_sem_mapa:
    print(f"[AVISO] Filiais NAO mapeadas: {filiais_sem_mapa}")
else:
    print("[OK] Todas as filiais mapeadas.")
print("Filiais:", df_custo["nm_pessoa_filial"].value_counts().to_dict())

[OK] Todas as filiais mapeadas.
Filiais: {'GSL PRUDENTE': 132, 'GSL MARINGA PR': 106, 'GSL DOURADOS': 35}


## Plano de Contas e classificacoes extras

Todas as linhas usam a conta **`6.6.1 FRETE DE TERCEIROS`** (provisorio — revisar se houver conta especifica para frete de motorista autonomo).

Classificacoes fixas (campos extras alem do modelo ODBC):

| Campo         | Valor       |
|---------------|-------------|
| `C/D`         | Custo       |
| `F/V`         | Variavel    |
| `D/I`         | Indireto    |
| `Natureza`    | Operacional |
| `Tipo_C_D_DFC`| Operacional |
| `Origem`      | Saidas (Aplicacoes) |

In [3]:
COD_CONTA   = "6.6.1"
DESC_CONTA  = "FRETE DE TERCEIROS"

# Classificacoes fixas do DFC
CLASSIF_CD         = "Custo"
CLASSIF_FV         = "Variavel"
CLASSIF_DI         = "Indireto"
CLASSIF_NATUREZA   = "Operacional"
CLASSIF_TIPO_DFC   = "Operacional"
ORIGEM             = "Saidas (Aplicacoes)"
SISTEMA            = "ATUA"

print(f"Conta: {COD_CONTA} {DESC_CONTA}")
print(f"Classificacao: C/D={CLASSIF_CD} | F/V={CLASSIF_FV} | D/I={CLASSIF_DI} | Natureza={CLASSIF_NATUREZA} | Tipo_C_D_DFC={CLASSIF_TIPO_DFC}")

Conta: 6.6.1 FRETE DE TERCEIROS
Classificacao: C/D=Custo | F/V=Variavel | D/I=Indireto | Natureza=Operacional | Tipo_C_D_DFC=Operacional


## Conversao para FECHAMENTO_ODBC

O `valor_conta` (e `valor_nf`, `valor_pago`) recebe `vl_frete_motorista` como valor **negativo**, conforme especificado para Saidas (Aplicacoes).

In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str((Path.cwd().parent / "Utitlities").resolve()))
from fechamento_excel import parse_valor_fechamento, parse_data_fechamento, gravar_fechamento_excel

modelo_cols = pd.read_excel(ARQUIVO_MODELO, nrows=0).columns.tolist()
# Colunas extras de classificacao (alem do modelo ODBC)
cols_extras = ["C/D", "F/V", "D/I", "Natureza", "Tipo_C_D_DFC"]
colunas_saida = modelo_cols + cols_extras

filiais_nao_mapeadas = []
linhas_saida = []

for _, row in df_custo.iterrows():
    filial_raw = _str(row.get("nm_pessoa_filial", ""))
    cc = mapear_cc_custo(filial_raw)
    if cc is None:
        filiais_nao_mapeadas.append(filial_raw)

    ctrc      = _str(row.get("nr_ctrc", ""))
    valor_raw = _to_float(row.get("vl_frete_motorista", 0))
    valor_neg = -abs(valor_raw)           # sempre negativo
    motorista = _str(row.get("nm_pessoa_motorista", ""))

    nova = {col: "" for col in colunas_saida}

    nova["filial"]               = cc["filial_saida"] if cc else filial_raw
    nova["titulo"]               = str(ctrc)
    nova["credor_forn_cli_func"] = motorista
    nova["data_nf"]              = parse_data_fechamento(row.get("dt_emissao", ""))
    nova["data_pagamento"]       = parse_data_fechamento(row.get("dt_emissao", ""))
    nova["valor_nf"]             = valor_neg
    nova["valor_pago"]           = -abs(valor_neg) if valor_neg is not None else pd.NA
    nova["valor_conta"]          = -abs(valor_neg) if valor_neg is not None else pd.NA
    nova["cod_conta"]            = COD_CONTA
    nova["conta"]                = DESC_CONTA
    nova["cod_conta-descr"]      = f"{COD_CONTA} {DESC_CONTA}"
    nova["observacao"]           = "frete motorista"
    nova["Origem"]               = ORIGEM
    nova["Sistema"]              = SISTEMA

    if cc:
        nova["Segmento"]              = cc["segmento"]
        nova["n1_cod_centro_custo"]   = cc["n1_cod"]
        nova["n1_centro_custo"]       = cc["n1_desc"]
        nova["n1_CC"]                 = f"{cc['n1_cod']} {cc['n1_desc']}"
        nova["n2_cod_centro_custo"]   = cc["n2_cod"]
        nova["n2_centro_custo"]       = cc["n2_desc"]
        nova["n2_CC"]                 = f"{cc['n2_cod']} {cc['n2_desc']}"
        nova["n3_cod_centro_custo"]   = cc["n3_cod"]
        nova["n3_centro_custo"]       = cc["n3_desc"]
        nova["n3_CC"]                 = f"{cc['n3_cod']} {cc['n3_desc']}"
        nova["n4_cod_centro_custo"]   = cc["n4_cod"]
        nova["n4_centro_custo"]       = cc["n4_desc"]
        nova["n4_CC"]                 = f"{cc['n4_cod']} {cc['n4_desc']}"

    nova["C/D"]          = CLASSIF_CD
    nova["F/V"]          = CLASSIF_FV
    nova["D/I"]          = CLASSIF_DI
    nova["Natureza"]     = CLASSIF_NATUREZA
    nova["Tipo_C_D_DFC"] = CLASSIF_TIPO_DFC

    linhas_saida.append(nova)

fechamento_df = pd.DataFrame(linhas_saida, columns=colunas_saida)

print(f"Linhas base ATUA: {len(df_ctrc)}")
print(f"Linhas geradas: {len(fechamento_df)}")
if len(fechamento_df) != len(df_ctrc):
    raise ValueError(
        f"Divergencia de linhas: base={len(df_ctrc)} vs fechamento={len(fechamento_df)}."
    )
if filiais_nao_mapeadas:
    print(f"[AVISO] Filiais sem mapeamento: {set(filiais_nao_mapeadas)}")
else:
    print("[OK] Todas as filiais mapeadas.")
print()
fechamento_df[["filial", "titulo", "credor_forn_cli_func", "data_nf", "valor_conta", "cod_conta", "conta", "n4_CC", "C/D", "F/V", "D/I", "Origem"]].head(5)

Linhas base ATUA: 273
Linhas geradas: 273
[OK] Todas as filiais mapeadas.



,filial,titulo,credor_forn_cli_func,data_nf,valor_conta,cod_conta,conta,n4_CC,C/D,F/V,D/I,Origem
0,GSL PRUDENTE,1029,ULISSES SANCHES LOPES,2026-01-05,-10080.0,6.6.1,FRETE DE TERCEIROS,1.4.1.1 TRANSPORTE,Custo,Variavel,Indireto,Saidas (Aplicacoes)
1,GSL PRUDENTE,1030,JOEL LEITE DE MORAIS,2026-01-05,-10080.0,6.6.1,FRETE DE TERCEIROS,1.4.1.1 TRANSPORTE,Custo,Variavel,Indireto,Saidas (Aplicacoes)
2,GSL PRUDENTE,1031,WILLIAN AMBROSIO,2026-01-05,-10080.0,6.6.1,FRETE DE TERCEIROS,1.4.1.1 TRANSPORTE,Custo,Variavel,Indireto,Saidas (Aplicacoes)
3,GSL DOURADOS,200,DAVID DE SOUZA FREITAS,2026-04-05,-9175.6,6.6.1,FRETE DE TERCEIROS,1.4.2.1 TRANSPORTE,Custo,Variavel,Indireto,Saidas (Aplicacoes)
4,GSL DOURADOS,201,JOSE ROBERTO VICENTE,2026-04-05,-15251.2,6.6.1,FRETE DE TERCEIROS,1.4.2.1 TRANSPORTE,Custo,Variavel,Indireto,Saidas (Aplicacoes)


## Salvamento

In [5]:
import sys
from pathlib import Path
sys.path.insert(0, str((Path.cwd().parent / "Utitlities").resolve()))
from fechamento_excel import parse_valor_fechamento, parse_data_fechamento, gravar_fechamento_excel

import datetime

SHEET_NAME = 'ATUA_custo_frete'

arquivo_saida_exec = ARQUIVO_SAIDA
try:
    gravar_fechamento_excel(fechamento_df, arquivo_saida_exec, sheet_name=SHEET_NAME)
    print(f"Arquivo gerado: {arquivo_saida_exec.resolve()}")
    print(f"Linhas gravadas: {len(fechamento_df)}")
except PermissionError:
    ts = datetime.datetime.now().strftime("%H%M%S")
    alt = arquivo_saida_exec.with_stem(f"{arquivo_saida_exec.stem}_{ts}")
    gravar_fechamento_excel(fechamento_df, alt, sheet_name=SHEET_NAME)
    print(f"[AVISO] Arquivo principal em uso. Salvo como: {alt.resolve()}")
    print(f"Linhas gravadas: {len(fechamento_df)}")

print()
if filiais_nao_mapeadas:
    print(f"[PENDENTE] Filiais sem mapa ({len(set(filiais_nao_mapeadas))}):")
    for f in sorted(set(filiais_nao_mapeadas)):
        print(f"  '{f}'")
else:
    print("[OK] Nenhuma pendencia. Todas as filiais foram mapeadas.")

Arquivo gerado: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\ATUA\Maio\ATUA_custo_frete_fechamento_05-2026.xlsx
Linhas gravadas: 273

[OK] Nenhuma pendencia. Todas as filiais foram mapeadas.
